# Emotion Recognition from Speech
### CodeAlpha Machine Learning Internship
**Name:** Sumit Kumar Mahto &nbsp;|&nbsp; **ID:** CA/DF1/47253

This one is really interesting — the goal is to listen to someone's voice and figure out if they're happy, sad, angry or neutral. 

The key idea is that we can't just feed raw audio into a neural network. We first convert it into something called MFCCs (Mel-Frequency Cepstral Coefficients), which basically represent the "texture" of the sound numerically. Then we train a CNN on those features.


In [ ]:
# uncomment to install if needed
# !pip install librosa tensorflow scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import os, warnings
warnings.filterwarnings("ignore")
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Dropout, Conv1D, MaxPooling1D,
                                      Flatten, BatchNormalization)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"tensorflow: {tf.__version__}")
print("good to go!")

## Step 1 — What do MFCCs look like?

Before jumping into training, let me visualise what we're actually feeding to the model.

In [ ]:
# generate a simple tone to demonstrate mfcc extraction
sr = 22050
t  = np.linspace(0, 2, sr * 2)
y_demo = (np.sin(2 * np.pi * 440 * t) + 0.3 * np.random.randn(len(t))).astype(np.float32)

mfccs_demo = librosa.feature.mfcc(y=y_demo, sr=sr, n_mfcc=40)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y_demo[:sr // 4], color="#7C4DFF", lw=0.6, alpha=0.8)
axes[0].set_title("raw audio waveform (first 0.25s)")
axes[0].set_xlabel("sample index")
axes[0].set_ylabel("amplitude")

librosa.display.specshow(mfccs_demo, sr=sr, x_axis="time", ax=axes[1], cmap="magma")
axes[1].set_title("MFCC features (40 coefficients over time)")
axes[1].set_ylabel("MFCC index")
plt.colorbar(axes[1].collections[0], ax=axes[1])
plt.tight_layout()
plt.show()

print(f"mfcc shape: {mfccs_demo.shape}  →  40 coefficients × time frames")

## Step 2 — Feature Extraction

This is the function that does the heavy lifting. For each audio file it extracts MFCCs, Chroma and Mel features and flattens them into one feature vector.

> **To use real RAVDESS data:** download it from Kaggle, set `data_path` to your folder and call `load_real_data(data_path)`

In [ ]:
def extract_features(file_path, n_mfcc=40):
    y, sr = librosa.load(file_path, duration=3, offset=0.5)

    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    mel    = librosa.feature.melspectrogram(y=y, sr=sr)

    return np.hstack([
        np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
        np.mean(chroma, axis=1),
        np.mean(mel, axis=1)[:20]
    ])

def load_real_data(data_path):
    # ravdess filename: 03-01-{emotion_code}-...-{actor}.wav
    emotion_map = {
        1: "neutral", 2: "calm", 3: "happy", 4: "sad",
        5: "angry",   6: "fearful", 7: "disgust", 8: "surprised"
    }
    X, labels = [], []
    for root, _, files in os.walk(data_path):
        for fname in files:
            if fname.endswith(".wav"):
                code  = int(fname.split("-")[2])
                label = emotion_map[code]
                feat  = extract_features(os.path.join(root, fname))
                X.append(feat)
                labels.append(label)
    return np.array(X), np.array(labels)

# --- using synthetic data until you get the real dataset ---
np.random.seed(42)
emotions    = ["neutral", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
n_each      = 200
feature_dim = 40 * 2 + 12 + 20   # 132 total

X = np.random.randn(n_each * len(emotions), feature_dim).astype(np.float32)
for i, em in enumerate(emotions):
    X[i * n_each:(i+1) * n_each, i*5:(i+1)*5] += np.random.randn(n_each, 5) * 2 + i

y_raw = np.repeat(emotions, n_each)
print(f"X shape  : {X.shape}")
print(f"emotions : {pd.Series(y_raw).value_counts().to_dict()}")

## Step 3 — Encode Labels & Split

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)
n_classes = len(le.classes_)
print(f"classes ({n_classes}): {list(le.classes_)}")

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# reshape for Conv1D — needs (samples, steps, channels)
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn  = X_test.reshape(X_test.shape[0],   X_test.shape[1],  1)

y_train_cat = to_categorical(y_train, n_classes)
y_test_cat  = to_categorical(y_test,  n_classes)

print(f"train : {X_train_cnn.shape}")
print(f"test  : {X_test_cnn.shape}")

## Step 4 — Build the CNN

Two convolutional blocks to extract patterns from the MFCC features, then a dense classifier on top.

In [ ]:
def build_model(input_shape, n_classes):
    model = Sequential([
        Conv1D(64, 5, activation="relu", input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),

        Conv1D(128, 5, activation="relu"),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),

        Flatten(),
        Dense(256, activation="relu"),
        Dropout(0.4),
        Dense(n_classes, activation="softmax")
    ])
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_model((X_train_cnn.shape[1], 1), n_classes)
model.summary()

## Step 5 — Train

In [ ]:
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=5, verbose=1)
]

history = model.fit(
    X_train_cnn, y_train_cat,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## Step 6 — Results

In [ ]:
# training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history["accuracy"],     label="train", color="#4CAF50", lw=2)
axes[0].plot(history.history["val_accuracy"], label="val",   color="#7C4DFF", lw=2)
axes[0].set_title("accuracy over epochs")
axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history["loss"],     label="train", color="#4CAF50", lw=2)
axes[1].plot(history.history["val_loss"], label="val",   color="#FF5722", lw=2)
axes[1].set_title("loss over epochs")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Emotion Recognition — Training Curves", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# confusion matrix
y_pred   = np.argmax(model.predict(X_test_cnn), axis=1)
test_acc = (y_pred == y_test).mean()
cm       = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f"confusion matrix  |  test accuracy: {test_acc:.2%}")
plt.ylabel("actual"); plt.xlabel("predicted")
plt.tight_layout()
plt.savefig("task2_confusion.png", dpi=150)
plt.show()

print(classification_report(y_test, y_pred, target_names=le.classes_))

## Step 7 — Save

In [ ]:
model.save("emotion_model.h5")
print("saved: emotion_model.h5")

# predict one sample
sample       = X_test_cnn[0:1]
predicted    = le.classes_[np.argmax(model.predict(sample))]
actual       = le.classes_[y_test[0]]
confidence   = model.predict(sample)[0].max()

print(f"
predicted emotion : {predicted}")
print(f"actual emotion    : {actual}")
print(f"confidence        : {confidence:.2%}")